# 🚀 Retail Data Pipeline — شرح الكود بالتفصيل

هذا الـ Notebook يشرح ملفات `utils.py` و `ingest.py` الموجودة في `airflow/tasks/` سطراً بسطر.

---

## 📁 هيكل المشروع

```
retail-data-pipeline/
├── airflow/
│   └── tasks/
│       ├── utils.py      ← دوال مساعدة (Helper Functions)
│       └── ingest.py     ← مهمة استيعاب البيانات (Ingestion Task)
├── data/raw/             ← ملفات CSV الخام
└── notebooks/            ← أنت هنا!
```

---

## 🧠 فكرة المشروع العامة

البايبلاين بتاعنا بيعمل الآتي:

1. **📥 Ingest**: بياخد ملفات CSV خام من الجهاز
2. **🔄 Convert**: بيحولها لـ Parquet (صيغة أسرع وأكفأ)
3. **☁️ Upload**: بيرفعها على Azure Blob Storage في طبقة Bronze

---

# 📄 الملف الأول: `utils.py`

ده الملف اللي فيه **الأدوات المشتركة** اللي بتستخدمها كل Tasks التانية.
فكره زي صندوق العدة 🧰 — بيجيبوه كل شغلانة محتاجة تشتغل مع Azure.

## 📦 الاستيرادات (Imports)

```python
# airflow/tasks/utils.py
import os, io
import pandas as pd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
```

| السطر | الشرح |
|-------|-------|
| `import os, io` | 🖥️ `os` للتعامل مع متغيرات البيئة — `io` لإنشاء Buffers في الذاكرة |
| `import pandas as pd` | 🐼 مكتبة pandas للتعامل مع البيانات الجدولية (DataFrames) |
| `from dotenv import load_dotenv` | 🔑 لتحميل المتغيرات السرية من ملف `.env` |
| `from azure.storage.blob import BlobServiceClient` | ☁️ مكتبة Azure للتواصل مع Blob Storage |

## ⚙️ تحميل الإعدادات

```python
load_dotenv()  # reads .env file automatically

AZURE_CONN_STR = os.getenv('AZURE_CONN_STR')
CONTAINER      = os.getenv('AZURE_CONTAINER', 'retail-pipeline')
```

### 🔍 شرح تفصيلي:

**`load_dotenv()`**
- 📂 بيفتح ملف `.env` الموجود في الـ root directory
- بيقرا كل السطور اللي فيها `KEY=VALUE`
- بيحطها في متغيرات البيئة تلقائياً
- ✅ فايدته: إننا مش بنكتب passwords أو connection strings جوا الكود مباشرةً

**`AZURE_CONN_STR = os.getenv('AZURE_CONN_STR')`**
- 🔐 بيجيب Connection String بتاع Azure من متغيرات البيئة
- Connection String هو نص طويل فيه عنوان الـ Storage Account + مفتاح الوصول
- لو المتغير مش موجود، القيمة هتبقى `None`

**`CONTAINER = os.getenv('AZURE_CONTAINER', 'retail-pipeline')`**
- 📦 بيجيب اسم الـ Container من متغيرات البيئة
- الـ `'retail-pipeline'` ده **قيمة افتراضية (default)** لو المتغير مش موجود
- Container في Azure زي الـ Folder الرئيسي اللي بيحتوي الملفات

## 🔌 الدالة الأولى: `get_blob_client()`

```python
def get_blob_client():
    return BlobServiceClient.from_connection_string(AZURE_CONN_STR)
```

### 🔍 شرح تفصيلي:

| الجزء | الشرح |
|-------|-------|
| `def get_blob_client():` | 🏗️ تعريف دالة اسمها `get_blob_client` بدون parameters |
| `BlobServiceClient.from_connection_string(...)` | 🔗 بينشئ اتصال بـ Azure باستخدام الـ Connection String |
| `return` | 📤 بيرجع الـ Client object اللي هيستخدمه الكود التاني |

### 💡 ليه عملنا دالة منفصلة؟
- **مبدأ DRY** (Don't Repeat Yourself) — بدل ما نكرر الكود في كل مكان
- لو Connection String اتغير، بنغيره في مكان واحد بس
- ممكن نضيف error handling بسهولة مستقبلاً

## 📥 الدالة الثانية: `read_parquet()`

```python
def read_parquet(blob_name: str) -> pd.DataFrame:
    '''Read a Parquet file from Azure Blob into a DataFrame.'''
    client = get_blob_client()
    blob   = client.get_blob_client(container=CONTAINER, blob=blob_name)
    data   = blob.download_blob().readall()
    return pd.read_parquet(io.BytesIO(data))
```

### 🔍 شرح سطر بسطر:

**`def read_parquet(blob_name: str) -> pd.DataFrame:`**
- 📝 بتعرف دالة بتاخد اسم الـ blob (نوعه `str`)
- `-> pd.DataFrame` ده **Type Hint** بيقول إن الدالة هترجع DataFrame
- Type Hints مش إجبارية لكنها بتساعد في فهم الكود

**`'''Read a Parquet file from Azure Blob into a DataFrame.'''`**
- 📖 Docstring — وصف مختصر لوظيفة الدالة

**`client = get_blob_client()`**
- 🔌 بيجيب الاتصال بـ Azure Blob Service كامل

**`blob = client.get_blob_client(container=CONTAINER, blob=blob_name)`**
- 🎯 بيحدد الملف المحدد اللي عايزه جوا الـ Container
- `container=CONTAINER` → اسم الـ Container (مثلاً `retail-pipeline`)
- `blob=blob_name` → اسم الملف (مثلاً `bronze/daily_sales.parquet`)

**`data = blob.download_blob().readall()`**
- ⬇️ `.download_blob()` بيبدأ تحميل الملف من Azure
- `.readall()` بيقرا كل محتوى الملف ويخزنه في الذاكرة كـ bytes

**`return pd.read_parquet(io.BytesIO(data))`**
- `io.BytesIO(data)` → بيحول الـ bytes لـ **virtual file في الذاكرة** (مش محتاجين نحفظه على الجهاز)
- `pd.read_parquet(...)` → بيقرا الـ Parquet من الـ virtual file ويحوله لـ DataFrame
- 🚀 الميزة: بنجيب البيانات من Azure مباشرة للذاكرة بدون ما نحفظ ملف مؤقت

## 📤 الدالة الثالثة: `write_parquet()`

```python
def write_parquet(df: pd.DataFrame, blob_name: str) -> None:
    '''Write a DataFrame as Parquet to Azure Blob.'''
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine='pyarrow')
    buffer.seek(0)
    client = get_blob_client()
    client.get_blob_client(container=CONTAINER, blob=blob_name) \
          .upload_blob(buffer, overwrite=True)
    print(f'[Azure] Written -> {blob_name} ({len(df):,} rows)')
```

### 🔍 شرح سطر بسطر:

**`def write_parquet(df: pd.DataFrame, blob_name: str) -> None:`**
- 🏗️ الدالة بتاخد DataFrame + اسم الـ blob
- `-> None` يعني الدالة مش بترجع قيمة (بس بتعمل Upload)

**`buffer = io.BytesIO()`**
- 💾 بيعمل **ملف وهمي (virtual file) في الذاكرة RAM**
- بدل ما نحفظ ملف على الهارد، بنكتب في الذاكرة مباشرةً
- أسرع وما بيحتاجش مساحة على الجهاز

**`df.to_parquet(buffer, index=False, engine='pyarrow')`**
- 🔄 بيحول الـ DataFrame لصيغة Parquet ويكتبه في الـ buffer
- `index=False` → مش بيحفظ index الـ DataFrame (بيوفر مساحة)
- `engine='pyarrow'` → بيستخدم مكتبة PyArrow لتحويل الصيغة (أسرع وأكثر توافقاً)

**`buffer.seek(0)`**
- ⏮️ بيرجع المؤشر لأول الـ buffer
- لازم نعمل ده لأن بعد الكتابة، المؤشر بيكون في الآخر
- لو ما عملناش seek(0)، هيرفع ملف فاضي!

**`client.get_blob_client(container=CONTAINER, blob=blob_name)`**
- 🎯 بيحدد المكان في Azure اللي هيتم الرفع فيه

**`.upload_blob(buffer, overwrite=True)`**
- ⬆️ بيرفع محتوى الـ buffer لـ Azure
- `overwrite=True` → لو الملف موجود قبل كده، بيستبدله (مش بيديك error)

**`print(f'[Azure] Written -> {blob_name} ({len(df):,} rows)')`**
- 🖨️ بيطبع رسالة تأكيد فيها اسم الملف وعدد الصفوف
- `{len(df):,}` → الـ `:,` بتضيف فاصلة للأرقام الكبيرة (مثلاً `1,000,000`)

---

# 📄 الملف الثاني: `ingest.py`

ده الملف اللي بينفذ مهمة الاستيعاب (Ingestion Task).
مهمته: بياخد ملفات CSV من الجهاز ويحولها لـ Parquet ويرفعها على Azure في **طبقة Bronze** 🥉

## 📦 الاستيرادات وإعداد الـ Path

```python
import pandas as pd
import sys, os
sys.path.insert(0, os.path.dirname(__file__))
from utils import write_parquet
```

| السطر | الشرح |
|-------|-------|
| `import pandas as pd` | 🐼 لقراءة ملفات CSV |
| `import sys, os` | 🖥️ `sys` لإدارة مسارات Python — `os` لمسارات الملفات |
| `sys.path.insert(0, os.path.dirname(__file__))` | 🗺️ بيضيف مجلد الملف الحالي لـ Python path حتى يقدر يستورد `utils` |
| `from utils import write_parquet` | 🔧 بيستورد دالة `write_parquet` من ملف `utils.py` |

### 💡 ليه `sys.path.insert`؟
Python بيدور على الـ modules في أماكن محددة.
لما نضيف مجلد الملف الحالي للـ path، بنقوله:
> "ابحث هنا كمان لما حد يعمل `import`"

`__file__` هو متغير خاص جوا Python بيحتوي على مسار الملف الحالي نفسه.

## 🗂️ قاموس المصادر: `SOURCES`

```python
SOURCES = {
    'bronze/daily_sales.parquet':      'data/raw/AU_Daily_Sales.csv',
    'bronze/sales_by_model.parquet':   'data/raw/AU_Sales_By_Model.csv',
    'bronze/car_models.parquet':       'data/raw/AU_Car_Models.csv',
    'bronze/dealers.parquet':          'data/raw/AU_Dealers.csv',
    'bronze/recalls.parquet':          'data/raw/AU_Car_Recalls.csv',
    'bronze/sentiment.parquet':        'data/raw/AU_Sentiment.csv',
}
```

### 🔍 شرح:
- **SOURCES** هو Dictionary (قاموس) بيعمل **mapping** بين:
  - 🔑 **Key**: اسم الملف على Azure (المسار في Blob Storage)
  - 📄 **Value**: مسار ملف CSV على الجهاز

### 📊 الملفات الـ 6:

| الملف على Azure | الملف المحلي | الوصف |
|----------------|-------------|-------|
| `bronze/daily_sales.parquet` | `AU_Daily_Sales.csv` | 📈 مبيعات يومية |
| `bronze/sales_by_model.parquet` | `AU_Sales_By_Model.csv` | 🚗 مبيعات حسب الموديل |
| `bronze/car_models.parquet` | `AU_Car_Models.csv` | 🏎️ بيانات الموديلات |
| `bronze/dealers.parquet` | `AU_Dealers.csv` | 🏪 بيانات الوكلاء |
| `bronze/recalls.parquet` | `AU_Car_Recalls.csv` | ⚠️ استدعاءات السيارات |
| `bronze/sentiment.parquet` | `AU_Sentiment.csv` | 💬 تحليل المشاعر |

### 🥉 ليه Bronze؟
المشروع بيستخدم **Medallion Architecture**:
```
Bronze 🥉 → Silver 🥈 → Gold 🥇
(Raw Data)  (Cleaned)   (Aggregated)
```
Bronze = البيانات الخام كما هي بدون أي معالجة

## ⚡ الدالة الرئيسية: `ingest_bronze()`

```python
def ingest_bronze():
    print('=' * 55)
    print('TASK: ingest_bronze -> CSV to Azure Blob (Bronze)')
    print('=' * 55)
    
    for blob_name, csv_path in SOURCES.items():
        df = pd.read_csv(csv_path)
        write_parquet(df, blob_name)
    print('\n[DONE] Bronze ingestion complete.')
```

### 🔍 شرح سطر بسطر:

**`def ingest_bronze():`**
- 🏗️ تعريف الدالة الرئيسية بدون parameters

**`print('=' * 55)`**
- 📏 بيطبع سطر من 55 علامة `=` كـ separator بصري في الـ logs
- `'=' * 55` في Python = كرر الحرف 55 مرة

**`print('TASK: ingest_bronze -> CSV to Azure Blob (Bronze)')`**
- 📢 رسالة بتوضح إن المهمة بدأت وإيه اللي بتعمله

**`for blob_name, csv_path in SOURCES.items():`**
- 🔄 Loop بيمشي على كل عنصر في الـ SOURCES dictionary
- `.items()` بترجع كل زوج (key, value) كـ tuple
- في كل iteration: `blob_name` = المسار على Azure، `csv_path` = المسار المحلي

**`df = pd.read_csv(csv_path)`**
- 📂 بيقرا ملف الـ CSV من الجهاز ويحوله لـ DataFrame
- pandas ذكي بما يكفي إنه يحدد أنواع البيانات تلقائياً

**`write_parquet(df, blob_name)`**
- ☁️ بيستدعي الدالة من `utils.py` لرفع الـ DataFrame على Azure
- بيحوله Parquet ويرفعه تلقائياً

**`print('\n[DONE] Bronze ingestion complete.')`**
- ✅ رسالة نجاح بعد الانتهاء من رفع كل الملفات

## 🚦 نقطة الدخول: `if __name__ == '__main__'`

```python
if __name__ == '__main__':
    ingest_bronze()
```

### 🔍 شرح:

ده من أهم الـ patterns في Python!

- `__name__` متغير خاص بيحتوي على اسم الـ module الحالي
- لو الملف اتشغل **مباشرةً**: `__name__ == '__main__'` ✅ → بيشغل `ingest_bronze()`
- لو الملف **اتستورد** من ملف تاني: `__name__ != '__main__'` ❌ → مش بيشغل الدالة

```
python ingest.py        → __name__ = '__main__'  → بيشغل الكود ✅
import ingest           → __name__ = 'ingest'    → ما بيشغلش الكود ❌
```

**ليه مهم؟** Airflow بيستورد الملفات ومش عايز الكود يشتغل تلقائياً عند الاستيراد!

---

# 🔄 تدفق البيانات الكامل

```
📂 data/raw/AU_Daily_Sales.csv
         │
         ▼
🐼 pd.read_csv()  →  DataFrame في الذاكرة
         │
         ▼
💾 io.BytesIO()   →  Virtual file في RAM
         │
         ▼
🔄 df.to_parquet() →  تحويل لصيغة Parquet
         │
         ▼
☁️ upload_blob()  →  Azure Blob Storage
         │
         ▼
📦 bronze/daily_sales.parquet على Azure
```

---

# 💡 ليه Parquet وليس CSV؟

| المعيار | CSV | Parquet |
|---------|-----|----------|
| 📦 الحجم | كبير | أصغر بـ 5-10x |
| ⚡ السرعة | بطيء | أسرع بكثير |
| 🔢 أنواع البيانات | نص فقط | يحفظ الأنواع |
| 📊 Column Pruning | لا | يقرا Columns محددة فقط |

---

# 🔐 الأمان

المشروع بيستخدم `.env` file لحفظ البيانات الحساسة:

```env
AZURE_CONN_STR=DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;
AZURE_CONTAINER=retail-pipeline
```

ملف `.gitignore` بيمنع رفع `.env` على GitHub 🛡️

---

# ✅ ملخص سريع

| الملف | الوظيفة |
|-------|---------|
| `utils.py` | 🧰 أدوات مشتركة: قراءة وكتابة Parquet من/إلى Azure |
| `ingest.py` | 📥 مهمة البايبلاين: CSV → Parquet → Azure Bronze Layer |